# Two-stage XGBoost: training on earlier elections, testing on 2019

This notebook documents the complete workflow: audit the data, construct pre-election features, tune and refit **two separate XGBoost models**, then evaluate on the **2019 test sample**. Only `TEST_TRAIN/train.csv` is read; 2024 is not used.

1. The first XGBoost learns **P(change | X)** from all eligible earlier seats.
2. The second XGBoost learns **P(destination | change, X)** from only earlier seats that actually changed.
3. Combine these probabilities to predict the winning party for every eligible 2019 seat, following the conditional construction in `06_exploring_multilayer_model.ipynb`.

The final training sample includes 1987, 1992, 1997, 2001, 2005, 2010, 2015 and 2017. Inner expanding-window validation selects each model's settings independently; 2019 outcomes are used only for evaluation. Scores below distinguish the first model's own binary accuracy, conditional destination accuracy, and combined winner accuracy. Historical validation scores are tuning diagnostics, not additional untouched test results.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "TEST_TRAIN" / "train.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the repository.")
raw = pd.read_csv(ROOT / "TEST_TRAIN" / "train.csv")
TEST_YEAR = 2019
EVALUATION_YEARS = [TEST_YEAR]
PARTIES = ["con", "lib", "lab", "natSW"]
ROLES = ["incumbent", "contesting_party", "third_party", "fourth_party"]
display(raw.groupby("election").size().rename("available_rows").to_frame())
from sklearn.metrics import (roc_auc_score, precision_score, recall_score,
                             brier_score_loss, confusion_matrix)
import sklearn, xgboost
print(f"scikit-learn {sklearn.__version__}; XGBoost {xgboost.__version__}")

,available_rows
election,
1987,633
1992,634
1997,641
2001,641
2005,628
2010,632
2015,632
2017,632
2019,632


scikit-learn 1.9.1; XGBoost 3.4.1


## Eligibility and role-based features

Previous winners labelled `oth` cannot be mapped because there is no corresponding previous share column. Exclude those rows, missing previous winners, missing outcome labels, and incomplete previous party shares (needed to rank challengers). **Keep current `oth` winners** when the previous winner is supported: these are valid seat changes. The audit below reports coverage by election.

`natSW` is the dataset's combined SNP/Plaid category. There are no polling or projected-share fields for this category. Missing numeric role-specific values are left as missing for XGBoost to handle natively. Existing projections are used as provided, without clipping or recomputing them. National-share features use the previous actual election’s national vote shares. Projections add the polling-minus-previous-national-vote-share change to previous constituency shares; they are not clipped to 0–1 and should not be interpreted as calibrated vote forecasts.

Current-election shares, majority and winner are never predictors. The previous winning party’s last-election vote share and region are retained. Party identities and the national governing party are categorical predictors.

In [2]:
previous_columns = [f"previous_{p}_share" for p in PARTIES]
known_previous = raw["previous_winner"].isin(PARTIES)
complete_previous = raw[previous_columns].notna().all(axis=1)
known_outcome = raw["winner"].notna()
eligible = known_previous & complete_previous & known_outcome
coverage = raw.assign(
    eligible=eligible,
    unsupported_previous=~known_previous,
    incomplete_previous=~complete_previous,
    missing_outcome=~known_outcome,
).groupby("election").agg(
    available=("eligible", "size"), included=("eligible", "sum"),
    unsupported_previous=("unsupported_previous", "sum"),
    incomplete_previous=("incomplete_previous", "sum"),
    missing_outcome=("missing_outcome", "sum"),
)
coverage["excluded"] = coverage["available"] - coverage["included"]
display(coverage)  # Exclusion reasons can overlap.
data = raw.loc[eligible].copy().reset_index(drop=True)


def engineer_features(frame):
    """Use only pre-election fields; preserve each party's role across families."""
    shares = frame[previous_columns].to_numpy(dtype=float)
    incumbent_index = pd.Index(PARTIES).get_indexer(frame["previous_winner"])
    if (incumbent_index < 0).any() or not np.isfinite(shares).all():
        raise ValueError("Role mapping needs supported previous winners and complete shares.")
    ranked = np.argsort(-shares, axis=1, kind="stable")
    challengers = ranked[ranked != incumbent_index[:, None]].reshape(-1, 3)
    role_indices = np.column_stack([incumbent_index, challengers])
    families = {
        "share": previous_columns,
        "projected_share": ["projected_con_share", "projected_lib_share", "projected_lab_share", None],
        "polling": ["con_polling", "lib_polling", "lab_polling", None],
        "previous_national_share": [f"previous_{p}_national_vote_share" for p in PARTIES],
    }
    features = pd.DataFrame(index=frame.index)
    for role_number, role in enumerate(ROLES):
        features[f"{role}_party"] = np.asarray(PARTIES)[role_indices[:, role_number]]
        for suffix, columns in families.items():
            values = np.column_stack([
                frame[col].to_numpy(dtype=float) if col else np.full(len(frame), np.nan)
                for col in columns
            ])
            features[f"{role}_{suffix}"] = values[np.arange(len(frame)), role_indices[:, role_number]]
    features["previous_share_gap"] = features["incumbent_share"] - features["contesting_party_share"]
    features["projected_share_gap"] = features["incumbent_projected_share"] - features["contesting_party_projected_share"]
    features["previous_winning_party_last_election_vote_share"] = frame["previous_winning_party_last_election_vote_share"]
    features["country/region"] = frame["country/region"]
    features["national_governing_party"] = frame["incumbent"]
    return features

X = engineer_features(data)
y = data["winner"].ne(data["previous_winner"]).astype(int).rename("seat_changed")
assert X["incumbent_party"].equals(data["previous_winner"].rename("incumbent_party"))
assert X["incumbent_party"].ne(X["contesting_party_party"]).all()
# Changing the outcome cannot change the features.
pd.testing.assert_frame_equal(X, engineer_features(data.drop(columns=["winner"])))


,available,included,unsupported_previous,incomplete_previous,missing_outcome,excluded
election,,,,,,
1987,633,633,0,0,0,0
1992,634,633,1,1,0,1
1997,641,550,91,91,0,91
2001,641,639,2,2,0,2
2005,628,626,2,2,0,2
2010,632,631,1,1,0,1
2015,632,630,2,0,0,2
2017,632,628,4,1,0,4
2019,632,629,2,3,0,3


## Chronological train/test split

Eligibility and features above retain the existing notebook definitions. The first stage uses all eligible training seats; the second uses their changed-seat subset. The table records the contribution of each election. Repeated constituencies across years are expected: these are constituency-election observations. No 2019 rows enter either model fit, preprocessing fit, or grid search.

In [3]:
training = data["election"].lt(TEST_YEAR)
evaluation = data["election"].eq(TEST_YEAR)
assert training.any() and evaluation.any()
assert data.loc[training, "election"].max() < TEST_YEAR
split_audit = data.assign(seat_changed=y).groupby("election").agg(
    eligible_seats=("winner", "size"), changed_seats=("seat_changed", "sum"))
split_audit["usage"] = np.where(split_audit.index < TEST_YEAR, "training", 
                               np.where(split_audit.index == TEST_YEAR, "test", "unused"))
display(split_audit)


,eligible_seats,changed_seats,usage
election,,,
1987,633,44,training
1992,633,51,training
1997,550,160,training
2001,639,22,training
2005,626,57,training
2010,631,111,training
2015,630,109,training
2017,628,65,training
2019,629,75,test


## Shared training procedure

For each stage, search the existing 16 combinations of `max_depth=[2,3,4,5]` and `n_estimators=[25,50,100,200]`, with `learning_rate=0.05`. Each inner fold trains on elections strictly before its validation election (1992 through 2017). Maximize mean validation accuracy with equal weight per election; tied scores use grid order. Refit the selected pipeline on all eligible pre-2019 observations for that stage.

Categorical imputation and dense one-hot encoding are fitted separately inside each fold. Numeric missing values remain missing for XGBoost. No class weighting or early stopping is used. The seed is 42, tree method is `hist`, and one thread is used. Binary training uses logistic loss; multiclass training uses soft probabilities and multiclass log loss. Labels are encoded using only classes present in each training fold; a single-class fold gives a constant prediction. Accuracy is the selection criterion, so these probabilities are not explicitly calibrated.

The following tables retain all grid scores and the selected candidate's accuracy in each historical validation election. Those fold scores participated in selection and should not be interpreted as unbiased test accuracies.

In [4]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.validation import check_is_fitted
from xgboost import XGBClassifier


class FoldEncodedXGB(ClassifierMixin, BaseEstimator):
    """Encode only the classes observed in this training fold."""
    def __init__(self, max_depth=3, n_estimators=100):
        self.max_depth = max_depth
        self.n_estimators = n_estimators

    def fit(self, X, y):
        self.encoder_ = LabelEncoder().fit(y)
        self.classes_ = self.encoder_.classes_
        self.n_features_in_ = X.shape[1]
        self.model_ = None
        if len(self.classes_) > 1:
            objective_params = (
                {"objective": "binary:logistic", "eval_metric": "logloss"}
                if len(self.classes_) == 2 else
                {"objective": "multi:softprob", "num_class": len(self.classes_),
                 "eval_metric": "mlogloss"}
            )
            self.model_ = XGBClassifier(
                max_depth=self.max_depth, n_estimators=self.n_estimators,
                learning_rate=0.05, **objective_params,
                tree_method="hist", n_jobs=1, random_state=42,
            )
            self.model_.fit(X, self.encoder_.transform(y))
        return self

    def predict(self, X):
        check_is_fitted(self, "classes_")
        if self.model_ is None:
            return np.repeat(self.classes_[0], X.shape[0])
        return self.encoder_.inverse_transform(self.model_.predict(X).astype(int))


    def predict_proba(self, X):
        check_is_fitted(self, "classes_")
        if self.model_ is None:
            return np.ones((X.shape[0], 1))
        # Normalize float32 softmax roundoff in float64 before probability scoring.
        probabilities = self.model_.predict_proba(X).astype(float)
        return probabilities / probabilities.sum(axis=1, keepdims=True)


def make_xgb_model():
    categorical = X.select_dtypes(include=["object", "string"]).columns.tolist()
    numeric = X.columns.difference(categorical).tolist()
    preprocessing = ColumnTransformer([
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numeric", "passthrough", numeric),
    ])
    return Pipeline([("preprocess", preprocessing), ("classifier", FoldEncodedXGB())])


PARAM_GRID = {
    "classifier__max_depth": [2, 3, 4, 5],
    "classifier__n_estimators": [25, 50, 100, 200],
}

validation_fold_rows = []
def tune_stage(mask, target, stage):
    train_X = X.loc[mask].reset_index(drop=True)
    train_y = target.loc[mask].reset_index(drop=True)
    years = data.loc[mask, "election"].reset_index(drop=True)
    splits = []
    for year in sorted(years.unique())[1:]:
        fit_idx = np.flatnonzero(years.lt(year))
        valid_idx = np.flatnonzero(years.eq(year))
        assert years.iloc[fit_idx].max() < year < TEST_YEAR
        splits.append((fit_idx, valid_idx))
        validation_fold_rows.append(dict(stage=stage, validation_election=year,
                                        n_train=len(fit_idx), n_validation=len(valid_idx)))
    search = GridSearchCV(make_xgb_model(), PARAM_GRID, scoring="accuracy",
                          cv=splits, refit=True, n_jobs=1, error_score="raise")
    search.fit(train_X, train_y)
    return search


def search_report(search, stage):
    scores = pd.DataFrame(search.cv_results_)
    display(scores[["param_classifier__max_depth", "param_classifier__n_estimators",
                    "mean_test_score", "std_test_score", "rank_test_score"]]
            .sort_values("rank_test_score").reset_index(drop=True))
    folds = pd.DataFrame(validation_fold_rows).query("stage == @stage").copy()
    folds["selected_model_validation_accuracy"] = [
        search.cv_results_[f"split{i}_test_score"][search.best_index_]
        for i in range(len(folds))]
    display(folds)
    print("Selected parameters:", search.best_params_)
    print(f"Mean earlier-election validation accuracy: {search.best_score_:.4%}")


## First XGBoost: does the seat change?

Fit on **all eligible pre-2019 seats**, with `seat_changed = (winner != previous_winner)`. Its standalone prediction uses a fixed threshold of **0.5**. Accuracy counts correct holds and correct changes; the always-hold baseline shows the effect of class imbalance. Change precision and recall, ROC-AUC and binary Brier score provide additional context. Brier score is lower-is-better; the other metrics are higher-is-better.

The confusion matrix has actual outcomes in rows and predictions in columns. The threshold is used only to evaluate this binary model; the final party probabilities use its unthresholded output.

In [5]:
change_search = tune_stage(training, y, "change")
change_model = change_search.best_estimator_
models = {TEST_YEAR: change_model}
search_report(change_search, "change")
positive_column = list(change_model.named_steps["classifier"].classes_).index(1)
change_probability = change_model.predict_proba(X.loc[evaluation])[:, positive_column]
predicted_change = (change_probability >= 0.5).astype(int)
actual = y.loc[evaluation]
first_stage_results = pd.DataFrame([{
    "election": TEST_YEAR, "n_train": int(training.sum()), "n_test": int(evaluation.sum()),
    "n_correct": int((actual.to_numpy() == predicted_change).sum()),
    "accuracy": accuracy_score(actual, predicted_change),
    "always_hold_accuracy": actual.eq(0).mean(),
    "precision": precision_score(actual, predicted_change, zero_division=0),
    "recall": recall_score(actual, predicted_change, zero_division=0),
    "roc_auc": roc_auc_score(actual, change_probability),
    "brier_score": brier_score_loss(actual, change_probability),
    "actual_change_rate": actual.mean(), "predicted_change_rate": predicted_change.mean(),
}]).set_index("election")
display(first_stage_results.round(4))
change_confusion = pd.DataFrame(confusion_matrix(actual, predicted_change, labels=[0, 1]),
    index=pd.Index(["hold", "change"], name="actual"),
    columns=pd.Index(["hold", "change"], name="predicted"))
assert change_confusion.to_numpy().sum() == evaluation.sum()
display(change_confusion)


,param_classifier__max_depth,param_classifier__n_estimators,mean_test_score,std_test_score,rank_test_score
0,3,100,0.892304,0.042448,1
1,4,100,0.889465,0.042064,2
2,4,200,0.889146,0.039776,3
3,2,50,0.888264,0.039495,4
4,3,50,0.887749,0.048190,5
5,2,100,0.887590,0.036729,6
6,3,200,0.886560,0.040342,7
7,2,200,0.885504,0.035321,8
8,4,50,0.884731,0.052725,9
9,5,200,0.881534,0.045758,10


,stage,validation_election,n_train,n_validation,selected_model_validation_accuracy
0,change,1992,633,633,0.925750
1,change,1997,1266,550,0.834545
2,change,2001,1816,639,0.946792
3,change,2005,2455,626,0.921725
4,change,2010,3081,631,0.890650
5,change,2015,3712,630,0.826984
6,change,2017,4342,628,0.899682


Selected parameters: {'classifier__max_depth': 3, 'classifier__n_estimators': 100}
Mean earlier-election validation accuracy: 89.2304%


,n_train,n_test,n_correct,accuracy,always_hold_accuracy,precision,recall,roc_auc,brier_score,actual_change_rate,predicted_change_rate
election,,,,,,,,,,,
2019,4970,629,573,0.911,0.8808,0.7436,0.3867,0.9307,0.0657,0.1192,0.062


predicted,hold,change
actual,,
hold,544,10
change,46,29


## Second XGBoost: who wins conditional on a change?

Only **actual changed seats before 2019** train this model. The target is the winning challenger role, mapped using previous results. `oth` remains a valid destination; roles absent from a training fold receive zero probability. The count table exposes rare classes.

The second stage produces conditional probabilities for **every eligible 2019 seat**, without using its actual outcome or requiring the first stage to predict a change. Its standalone evaluation uses only actual 2019 changes and includes an always-contesting-party baseline. This conditional accuracy assumes a change occurred and does not measure detection.

### Combining the probabilities

For incumbent party $i$ and any other party $p$:

$$P(W=i\mid X)=1-P(C=1\mid X)$$
$$P(W=p\mid X)=P(C=1\mid X)P(W=p\mid C=1,X),\quad p\ne i.$$

The incumbent has zero conditional destination probability. Map challenger-role probabilities back to parties, multiply by the first-stage probability, and give the incumbent the remaining mass. The final winner is the party with greatest combined probability. This may differ from applying a hard change threshold first. Assertions below check that probabilities sum to one and that the incumbent probability is exactly the first-stage hold probability.

In [6]:
from sklearn.metrics import log_loss

DESTINATION_ROLES = ["contesting_party", "third_party", "fourth_party", "oth"]
PARTY_LABELS = PARTIES + ["oth"]

# Outcomes define training labels and evaluation subsets only.
destination_target = pd.Series(pd.NA, index=data.index, dtype="object")
for role in DESTINATION_ROLES[:-1]:
    matches = y.eq(1) & data["winner"].eq(X[f"{role}_party"])
    destination_target.loc[matches] = role
destination_target.loc[y.eq(1) & data["winner"].eq("oth")] = "oth"
if destination_target.loc[y.eq(1)].isna().any():
    raise ValueError("Some changed-seat winners cannot be mapped to a destination role.")


def party_probability_metrics(actual, probabilities):
    """Multiclass Brier is the mean sum of squared errors across parties (0–2)."""
    actual = np.asarray(actual)
    predicted = np.asarray(PARTY_LABELS)[probabilities.argmax(axis=1)]
    one_hot = (actual[:, None] == np.asarray(PARTY_LABELS)[None, :]).astype(float)
    # sklearn log_loss expects probability columns in sorted label order.
    sorted_columns = np.argsort(PARTY_LABELS)
    return {
        "accuracy": accuracy_score(actual, predicted),
        "log_loss": log_loss(actual, probabilities[:, sorted_columns],
                             labels=sorted(PARTY_LABELS)),
        "multiclass_brier": np.square(probabilities - one_hot).sum(axis=1).mean(),
    }


destination_models = {}
destination_training_counts = []
conditional_metric_rows = []
combined_metric_rows = []
conditional_frames = []
combined_frames = []

for year in EVALUATION_YEARS:
    changed_train = data["election"].lt(year) & y.eq(1)
    evaluation = data["election"].eq(year)
    evaluation_data = data.loc[evaluation]
    evaluation_features = X.loc[evaluation]
    train_target = destination_target.loc[changed_train]
    if train_target.nunique() < 2:
        raise ValueError(f"{year}: at least two destination classes are needed.")
    assert data.loc[changed_train, "election"].max() < year
    assert y.loc[changed_train].eq(1).all()
    destination_training_counts.append({
        "election": year,
        **train_target.value_counts().reindex(DESTINATION_ROLES, fill_value=0).to_dict(),
    })
    destination_search = tune_stage(changed_train, destination_target, "destination")
    search_report(destination_search, "destination")
    destination_model = destination_search.best_estimator_
    destination_models[year] = destination_model
    role_probabilities = pd.DataFrame(
        destination_model.predict_proba(evaluation_features),
        index=evaluation_data.index,
        columns=destination_model.named_steps["classifier"].classes_,
    ).reindex(columns=DESTINATION_ROLES, fill_value=0.0)

    conditional = np.zeros((len(evaluation_data), len(PARTY_LABELS)))
    row_indices = np.arange(len(evaluation_data))
    for role in DESTINATION_ROLES[:-1]:
        party_indices = pd.Index(PARTY_LABELS).get_indexer(evaluation_features[f"{role}_party"])
        conditional[row_indices, party_indices] += role_probabilities[role].to_numpy()
    conditional[:, PARTY_LABELS.index("oth")] = role_probabilities["oth"].to_numpy()
    incumbent_indices = pd.Index(PARTY_LABELS).get_indexer(evaluation_data["previous_winner"])
    assert np.allclose(conditional[row_indices, incumbent_indices], 0)
    assert np.allclose(conditional.sum(axis=1), 1)

    # The first stage was already fitted on all eligible earlier seats.
    change_class = list(models[year].named_steps["classifier"].classes_).index(1)
    change_probability = models[year].predict_proba(evaluation_features)[:, change_class]
    combined = conditional * change_probability[:, None]
    combined[row_indices, incumbent_indices] = 1 - change_probability
    assert np.allclose(combined.sum(axis=1), 1)
    assert np.allclose(combined[row_indices, incumbent_indices], 1 - change_probability)
    assert ((combined >= 0) & (combined <= 1)).all()

    identifiers = evaluation_data[["constituency_name", "election", "previous_winner", "winner"]].copy()
    conditional_frame = identifiers.copy()
    combined_frame = identifiers.assign(change_probability=change_probability)
    for column, party in enumerate(PARTY_LABELS):
        conditional_frame[f"p_{party}_given_change"] = conditional[:, column]
        combined_frame[f"p_{party}"] = combined[:, column]
    conditional_frame["predicted_destination"] = np.asarray(PARTY_LABELS)[conditional.argmax(axis=1)]
    combined_frame["predicted_winner"] = np.asarray(PARTY_LABELS)[combined.argmax(axis=1)]
    conditional_frames.append(conditional_frame)
    combined_frames.append(combined_frame)

    actually_changed = y.loc[evaluation].eq(1).to_numpy()
    conditional_metrics = (
        party_probability_metrics(evaluation_data.loc[actually_changed, "winner"], conditional[actually_changed])
        if actually_changed.any()
        else dict.fromkeys(["accuracy", "log_loss", "multiclass_brier"], np.nan)
    )
    conditional_metric_rows.append({
        "election": year, "n_changed_train": int(changed_train.sum()),
        "n_changed_evaluation": int(actually_changed.sum()), **conditional_metrics,
    })
    combined_metric_rows.append({
        "election": year, "n_evaluation": len(evaluation_data),
        **party_probability_metrics(evaluation_data["winner"], combined),
        "changed_seat_accuracy": (
            accuracy_score(
                evaluation_data.loc[actually_changed, "winner"],
                combined_frame.loc[actually_changed, "predicted_winner"],
            ) if actually_changed.any() else np.nan
        ),
    })

conditional_party_probabilities = pd.concat(conditional_frames, ignore_index=True)
combined_party_probabilities = pd.concat(combined_frames, ignore_index=True)
conditional_results = pd.DataFrame(conditional_metric_rows).set_index("election")
combined_results = pd.DataFrame(combined_metric_rows).set_index("election")
destination_training_counts = pd.DataFrame(destination_training_counts).set_index("election")


,param_classifier__max_depth,param_classifier__n_estimators,mean_test_score,std_test_score,rank_test_score
0,3,50,0.906916,0.062054,1
1,2,25,0.902353,0.054310,2
2,3,25,0.901460,0.053670,3
3,4,25,0.901460,0.053670,3
4,5,25,0.901460,0.053670,3
5,5,50,0.892923,0.087894,6
6,4,50,0.891636,0.086771,7
7,2,50,0.889938,0.085230,8
8,5,100,0.877344,0.091088,9
9,3,100,0.876149,0.087865,10


,stage,validation_election,n_train,n_validation,selected_model_validation_accuracy
7,destination,1992,44,51,0.960784
8,destination,1997,95,160,0.956250
9,destination,2001,255,22,0.909091
10,destination,2005,277,57,0.877193
11,destination,2010,334,111,0.972973
12,destination,2015,445,109,0.779817
13,destination,2017,554,65,0.892308


Selected parameters: {'classifier__max_depth': 3, 'classifier__n_estimators': 50}
Mean earlier-election validation accuracy: 90.6916%


## 2019 results: conditional and complete model

The destination training counts contain **earlier elections only**. Conditional accuracy scores the new winner on actual changed test seats. Combined accuracy scores the final winner on all eligible test seats; combined `changed_seat_accuracy` also counts missed changes as errors. The always-hold party baseline predicts the previous winner.

Log loss and multiclass Brier score assess probability quality and are lower-is-better. Multiclass Brier sums squared errors across parties (range 0–2); it differs from the binary Brier scale above. An unseen class receives probability zero, clipped internally when computing log loss. These results apply only to eligible rows; excluded seats are not silently counted as correct.

In [7]:
display(destination_training_counts)
conditional_results["always_contesting_accuracy"] = accuracy_score(
    data.loc[evaluation & y.eq(1), "winner"], X.loc[evaluation & y.eq(1), "contesting_party_party"])
combined_results["always_hold_accuracy"] = accuracy_score(
    data.loc[evaluation, "winner"], data.loc[evaluation, "previous_winner"])
display(conditional_results.round(4))
display(combined_results.round(4))
print(f"First XGBoost binary accuracy: {first_stage_results.loc[TEST_YEAR, 'accuracy']:.2%}")
print(f"Second XGBoost conditional accuracy: {conditional_results.loc[TEST_YEAR, 'accuracy']:.2%}")
print(f"Combined winner accuracy: {combined_results.loc[TEST_YEAR, 'accuracy']:.2%}")
validation_folds = pd.DataFrame(validation_fold_rows)
assert len(combined_party_probabilities) == evaluation.sum()
assert destination_training_counts.to_numpy().sum() == (training & y.eq(1)).sum()


,contesting_party,third_party,fourth_party,oth
election,,,,
2019,554,44,11,10


,n_changed_train,n_changed_evaluation,accuracy,log_loss,multiclass_brier,always_contesting_accuracy
election,,,,,,
2019,619,75,0.9467,0.1205,0.0572,0.9867


,n_evaluation,accuracy,log_loss,multiclass_brier,changed_seat_accuracy,always_hold_accuracy
election,,,,,,
2019,629,0.9078,0.2236,0.1332,0.36,0.8808


First XGBoost binary accuracy: 91.10%
Second XGBoost conditional accuracy: 94.67%
Combined winner accuracy: 90.78%


### Interpreting the saved 2019 run

The first XGBoost trains on **4,970** earlier observations and selects depth **3** with **100** trees. Its mean historical validation accuracy is **89.23%**. On **629** eligible 2019 seats it correctly classifies **573 (91.10%)**, compared with **88.08%** for always predicting a hold. It correctly identifies 544 holds and 29 changes, with 10 false change alarms and 46 missed changes. Change precision is **74.36%** and recall is **38.67%**: the overall accuracy is dominated by holds.

The second XGBoost trains on **619** earlier changed seats and selects depth **3** with **50** trees. It predicts the destination correctly for **71 of 75 actual 2019 changes (94.67%)**. The always-contesting-party baseline achieves **74/75 (98.67%)**, so the fitted destination model does not outperform that baseline in this test election.

The combined model predicts **571/629 winners (90.78%)** correctly, including **27/75 actual changes (36.00%)**. The conditional destination score assumes a change is known; the combined result also reflects missed changes from the first stage. These figures describe the saved run and are not used to retune either model.

The fitted models are `change_model` and `destination_models[2019]`; the searches are `change_search` and `destination_search`, including every candidate and fold score in `cv_results_`. `validation_folds` records the chronological splits. `first_stage_results` and `change_confusion` document the first model independently. `conditional_party_probabilities` and `combined_party_probabilities` retain constituency-level 2019 predictions for inspection; their `winner` columns are evaluation labels only.

No settings are selected from the 2019 scores. A high conditional accuracy need not translate into high change recall or combined changed-seat accuracy. Class imbalance, rare destinations and changes between elections limit what a single held-out election establishes.